In [10]:
import pygame, sys, os, tkinter as tk, ast, numpy as np, shutil
from tkinter import filedialog

pygame.init()
WIDTH, HEIGHT = 900, 700
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption('Obstacle Calibration | Images | Map')
BG=(30,30,35); BOX=(50,50,55); TEXT=(220,220,220); BLUE=(0,150,255); GREEN=(40,180,80); RED=(180,60,60)
font=pygame.font.SysFont('consolas',16); title_font=pygame.font.SysFont('consolas',24,bold=True); msg_font=pygame.font.SysFont('consolas',15)
OFFSET_MAPA_X=50; OFFSET_MAPA_Y=50; CANVAS_W=800; CANVAS_H=600; CAMPO_LARGO_M=16.54; CAMPO_ANCHO_M=8.07
CONFIG_FILE='planner_config.txt'; DOWNLOADS_PATH=os.path.join(os.path.expanduser('~'),'Downloads'); DEFAULT_CANCHAS=os.path.join(DOWNLOADS_PATH,'Images'); DEFAULT_OBSTACULOS=os.path.join(DOWNLOADS_PATH,'obstaculos','map_obstacles.txt'); DEFAULT_IMAGEN_NOMBRE='2026'
borde_min_x=0; borde_max_x=CANVAS_W; borde_min_y=0; borde_max_y=CANVAS_H

def cargar_configuracion():
    config={}
    if os.path.exists(CONFIG_FILE):
        try:
            with open(CONFIG_FILE,encoding='utf-8') as f:
                for line in f:
                    line=line.strip()
                    if line and not line.startswith('#') and '=' in line:
                        k,v=line.split('=',1); config[k]=v
        except Exception: pass
    return config

def guardar_configuracion(config):
    keys=['folder_imagenes','file_obstaculos','anio_competencia','nombre_imagen']
    try:
        with open(CONFIG_FILE,'w',encoding='utf-8') as f:
            f.write('# Images and map settings\n[DIRECTORIES]\n')
            for k in keys: f.write(f'{k}={config.get(k,"")}\n')
        return True
    except Exception: return False

def detectar_limites_cancha(surf):
    global borde_min_x,borde_max_x,borde_min_y,borde_max_y
    try:
        arr=pygame.surfarray.array3d(surf); white=(arr[:,:,0]>220)&(arr[:,:,1]>220)&(arr[:,:,2]>220); w,h=white.shape
        xs=np.where(white.sum(axis=1)>h*.30)[0]; ys=np.where(white.sum(axis=0)>w*.30)[0]
        borde_min_x,borde_max_x=(int(xs.min()),int(xs.max())) if len(xs)>=2 else (0,w)
        borde_min_y,borde_max_y=(int(ys.min()),int(ys.max())) if len(ys)>=2 else (0,h)
    except Exception: borde_min_x,borde_max_x,borde_min_y,borde_max_y=0,CANVAS_W,0,CANVAS_H

def m_a_px(x,y):
    aw=max(1,borde_max_x-borde_min_x); ah=max(1,borde_max_y-borde_min_y)
    return (int(round(OFFSET_MAPA_X+borde_min_x+(float(x)/CAMPO_LARGO_M)*aw)),int(round(OFFSET_MAPA_Y+borde_max_y-(float(y)/CAMPO_ANCHO_M)*ah)) )
def px_delta_a_m(dx,dy):
    return float(dx)*CAMPO_LARGO_M/max(1,borde_max_x-borde_min_x),-float(dy)*CAMPO_ANCHO_M/max(1,borde_max_y-borde_min_y)
def calcular_puntos_px(data):
    slope,intercept,xr,yr=data[:4]
    if abs(xr[0]-xr[1])<1e-5: a,b=(xr[0],yr[0]),(xr[0],yr[1])
    elif abs(yr[0]-yr[1])<1e-5: a,b=(xr[0],yr[0]),(xr[1],yr[0])
    else: a,b=(xr[0],slope*xr[0]+intercept),(xr[1],slope*xr[1]+intercept)
    return m_a_px(*a),m_a_px(*b)
def dist_punto_segmento(px,py,x1,y1,x2,y2):
    l2=(x1-x2)**2+(y1-y2)**2
    if not l2:return ((px-x1)**2+(py-y1)**2)**.5
    t=max(0,min(1,((px-x1)*(x2-x1)+(py-y1)*(y2-y1))/l2)); return ((px-(x1+t*(x2-x1)))**2+(py-(y1+t*(y2-y1)))**2)**.5
def mover_segmento_matematico(data,dx,dy):
    slope,intercept,xr,yr=data[:4]; vert=abs(xr[0]-xr[1])<1e-5; horiz=abs(yr[0]-yr[1])<1e-5
    xr=[v+dx for v in xr]; yr=[v+dy for v in yr]
    if vert:data[1]+=dx
    elif horiz:data[1]+=dy
    else:data[1]=intercept-slope*dx+dy
    data[2]=[min(xr),max(xr)]; data[3]=[min(yr),max(yr)]

def segment_endpoints_m(data):
    slope,intercept,xr,yr=data[:4]
    if abs(xr[0]-xr[1])<1e-5:return [(xr[0],yr[0]),(xr[0],yr[1])]
    if abs(yr[0]-yr[1])<1e-5:return [(xr[0],yr[0]),(xr[1],yr[0])]
    return [(xr[0],slope*xr[0]+intercept),(xr[1],slope*xr[1]+intercept)]

def set_segment_endpoints(data,pts):
    (x1,y1),(x2,y2)=pts
    if abs(x2-x1)<1e-5:
        data[0]=0.0; data[1]=x1; data[2]=[x1,x1]; data[3]=[min(y1,y2),max(y1,y2)]
    elif abs(y2-y1)<1e-5:
        data[0]=0.0; data[1]=y1; data[2]=[min(x1,x2),max(x1,x2)]; data[3]=[y1,y1]
    else:
        m=(y2-y1)/(x2-x1); b=y1-m*x1
        data[0]=m;data[1]=b;data[2]=[min(x1,x2),max(x1,x2)];data[3]=[min(y1,y2),max(y1,y2)]

def repair_line_continuity(segments, tolerance_m=0.50):
    """
    Repair gaps in obstacle outlines within the same obstacle group.

    - For axis-aligned fragments, extend horizontal/vertical endpoints to
      nearby perpendicular segments so the lines meet.
    - Also join collinear fragments when their gap is small.
    - Does not modify diagonal segments.
    """
    repairs = 0
    eps = 1e-5

    # Snapshot the geometry so edits are not used as candidates in the same pass.
    snapshots = []
    for item in segments:
        d = item["data"]
        p1, p2 = segment_endpoints_m(d)
        snapshots.append((item["group_id"], p1, p2))

    def orientation(p1, p2):
        if abs(p1[0] - p2[0]) <= eps:
            return "V"
        if abs(p1[1] - p2[1]) <= eps:
            return "H"
        return "D"

    # Work per group to avoid connecting unrelated obstacles.
    groups = {}
    for i, snap in enumerate(snapshots):
        groups.setdefault(snap[0], []).append(i)

    for gid, ids in groups.items():
        for i in ids:
            _, p1, p2 = snapshots[i]
            oi = orientation(p1, p2)
            if oi not in ("H", "V"):
                continue
            current = segment_endpoints_m(segments[i]["data"])
            changed = False

            for endpoint_idx in (0, 1):
                ep = current[endpoint_idx]
                candidates = []

                for j in ids:
                    if i == j:
                        continue
                    _, q1, q2 = snapshots[j]
                    oj = orientation(q1, q2)

                    # Perpendicular line: extend this endpoint to the crossing,
                    # only if the crossing is close to the endpoint.
                    if oi == "H" and oj == "V":
                        x = q1[0]
                        y = ep[1]
                        qmin, qmax = sorted((q1[1], q2[1]))
                        if qmin - tolerance_m <= y <= qmax + tolerance_m:
                            crossing = (x, y)
                            distance = abs(x - ep[0])
                            # A horizontal endpoint must be moved along X.
                            if distance <= tolerance_m:
                                candidates.append((distance, crossing))
                    elif oi == "V" and oj == "H":
                        y = q1[1]
                        x = ep[0]
                        qmin, qmax = sorted((q1[0], q2[0]))
                        if qmin - tolerance_m <= x <= qmax + tolerance_m:
                            crossing = (x, y)
                            distance = abs(y - ep[1])
                            # A vertical endpoint must be moved along Y.
                            if distance <= tolerance_m:
                                candidates.append((distance, crossing))

                    # Collinear segments: close a small gap by moving the
                    # closest endpoint toward the other segment's endpoint.
                    elif oi == oj and oi in ("H", "V"):
                        if oi == "H" and abs(p1[1]-q1[1]) <= tolerance_m:
                            if abs(ep[1]-q1[1]) <= tolerance_m:
                                for qp in (q1, q2):
                                    gap = abs(qp[0]-ep[0])
                                    if gap <= tolerance_m and gap > eps:
                                        candidates.append((gap, qp))
                        elif oi == "V" and abs(p1[0]-q1[0]) <= tolerance_m:
                            if abs(ep[0]-q1[0]) <= tolerance_m:
                                for qp in (q1, q2):
                                    gap = abs(qp[1]-ep[1])
                                    if gap <= tolerance_m and gap > eps:
                                        candidates.append((gap, qp))

                if candidates:
                    candidates.sort(key=lambda c: c[0])
                    current[endpoint_idx] = candidates[0][1]
                    changed = True

            if changed:
                set_segment_endpoints(segments[i]["data"], current)
                repairs += 1

    return repairs

def auto_ajustar_por_color(mouse_pos,mapa_surf,segmentos):
    try:
        mx,my=mouse_pos; rx,ry=mx-OFFSET_MAPA_X,my-OFFSET_MAPA_Y; arr=pygame.surfarray.array3d(mapa_surf)
        if rx<0 or rx>=arr.shape[0] or ry<0 or ry>=arr.shape[1]:return False,'Click outside image',None
        target=arr[rx,ry]; mask=np.all(np.abs(arr.astype(np.int16)-target.astype(np.int16))<25,axis=2); xs,ys=np.where(mask)
        if not len(xs):return False,'Color not detected',None
        near=(xs-rx)**2+(ys-ry)**2<90**2; xs,ys=xs[near],ys[near]
        if len(xs)<10:return False,'Color area too small',None
        cxv=float(xs.mean()+OFFSET_MAPA_X); cyv=float(ys.mean()+OFFSET_MAPA_Y); best=float('inf'); idx=-1
        for i,s in enumerate(segmentos):
            p1,p2=calcular_puntos_px(s['data']); d=dist_punto_segmento(mx,my,*p1,*p2)
            if d<best:best,idx=d,i
        if idx<0 or best>150:return False,'No mathematical obstacle nearby',None
        gid=segmentos[idx]['group_id']; centers=[]
        for s in segmentos:
            if s['group_id']==gid:
                p1,p2=calcular_puntos_px(s['data']); centers.append(((p1[0]+p2[0])/2,(p1[1]+p2[1])/2))
        if not centers:return False,'Group error',None
        cx=sum(p[0] for p in centers)/len(centers); cy=sum(p[1] for p in centers)/len(centers); dx,dy=px_delta_a_m(cxv-cx,cyv-cy)
        for s in segmentos:
            if s['group_id']==gid:mover_segmento_matematico(s['data'],dx,dy)
        return True,'Obstacle group aligned',gid
    except Exception as e:return False,f'Alignment error: {e}',None
def leer_obstaculos(filepath,year):
    active=[]
    if not os.path.exists(filepath):return active
    try:
        with open(filepath,encoding='utf-8') as f: lines=f.readlines()
        group=0
        for i,line in enumerate(lines):
            s=line.strip()
            if s.startswith('/* OBSTACLE'):group+=1;continue
            if not s or s.startswith('#') or s.startswith('/*'):continue
            a,b=line.find('['),line.rfind(']')
            if a<0 or b<=a:continue
            try:
                data=ast.literal_eval(line[a:b+1])
                if isinstance(data,list) and len(data)>=5 and isinstance(data[2],list) and str(year) in str(data[4]):active.append({'line_idx':i,'data':data,'group_id':group if group else i+10000})
            except Exception:pass
    except Exception:pass
    return active
def guardar_calibracion(filepath,segments):
    try:
        with open(filepath,encoding='utf-8') as f:lines=f.readlines()
        def fmt(v):return f'{v:.6f}' if isinstance(v,float) else str(v)
        for item in segments:
            i=item['line_idx'];d=item['data'];old=lines[i];a=old.find('[');b=old.rfind(']')
            new=f'[{fmt(d[0])}, {fmt(d[1])}, [{fmt(d[2][0])}, {fmt(d[2][1])}], [{fmt(d[3][0])}, {fmt(d[3][1])}], "{d[4]}"]'
            lines[i]=old[:a]+new+old[b+1:]
        with open(filepath,'w',encoding='utf-8') as f:f.writelines(lines)
        return True
    except Exception:return False
def button(rect,color,label,mouse):
    c=tuple(min(255,v+35) for v in color) if rect.collidepoint(mouse) else color; pygame.draw.rect(screen,c,rect,border_radius=7); s=font.render(label,True,(255,255,255));screen.blit(s,(rect.x+(rect.w-s.get_width())//2,rect.y+(rect.h-s.get_height())//2))

def main():
    cfg=cargar_configuracion(); folder=cfg.get('folder_imagenes',DEFAULT_CANCHAS); obs=cfg.get('file_obstaculos',DEFAULT_OBSTACULOS); year=cfg.get('anio_competencia','2026'); image_name=cfg.get('nombre_imagen',DEFAULT_IMAGEN_NOMBRE)
    state='IMAGES'; image=None; segments=[]; calibrated_path=''; msg=''; selected=None; edit_mode=True; running=True
    fields=[('Image folder',folder),('Obstacle TXT',obs),('Competition year',year),('Image filename',image_name)]
    rects=[pygame.Rect(260,160+i*75,520,38) for i in range(4)]; save_btn=pygame.Rect(260,480,150,48); load_btn=pygame.Rect(460,480,170,48); back_btn=pygame.Rect(20,20,120,40); save_map_btn=pygame.Rect(730,15,150,40)
    while running:
        mouse=pygame.mouse.get_pos(); screen.fill(BG)
        for e in pygame.event.get():
            if e.type==pygame.QUIT:running=False
            elif state=='IMAGES':
                if e.type==pygame.MOUSEBUTTONDOWN and e.button==1:
                    for i,r in enumerate(rects):
                        if r.collidepoint(e.pos):
                            if i==0:
                                root=tk.Tk();root.withdraw();v=filedialog.askdirectory(title='Select images folder');root.destroy();
                                if v:fields[i]=('Image folder',v)
                            elif i==1:
                                root=tk.Tk();root.withdraw();v=filedialog.askopenfilename(title='Select obstacle file',filetypes=[('Text files','*.txt')]);root.destroy();
                                if v:fields[i]=('Obstacle TXT',v)
                            else:fields[i]=(fields[i][0],fields[i][1])
                    if save_btn.collidepoint(e.pos):
                        cfg.update(folder_imagenes=fields[0][1],file_obstaculos=fields[1][1],anio_competencia=fields[2][1],nombre_imagen=fields[3][1]);msg='Settings saved' if guardar_configuracion(cfg) else 'Save failed'
                    elif load_btn.collidepoint(e.pos):
                        cfg.update(folder_imagenes=fields[0][1],file_obstaculos=fields[1][1],anio_competencia=fields[2][1],nombre_imagen=fields[3][1]); imgname=fields[3][1]; imgname=imgname if os.path.splitext(imgname)[1] else imgname+'.png';imgpath=os.path.join(fields[0][1],imgname);obspath=fields[1][1]
                        if os.path.isfile(imgpath) and os.path.isfile(obspath):
                            calibrated_path=obspath[:-4]+'_calibrated.txt' if obspath.lower().endswith('.txt') else obspath+'_calibrated.txt'
                            if not os.path.exists(calibrated_path):
                                try:shutil.copy2(obspath,calibrated_path)
                                except Exception:calibrated_path=obspath
                            try:image=pygame.transform.smoothscale(pygame.image.load(imgpath),(CANVAS_W,CANVAS_H));detectar_limites_cancha(image);segments=leer_obstaculos(calibrated_path,fields[2][1]);selected=None;state='MAP';msg='Left click to align an obstacle group by color. ESC returns.'
                            except Exception as ex:msg=f'Image load error: {ex}'
                        else:msg='Image or obstacle file not found.'
                elif e.type==pygame.KEYDOWN:
                    if e.key==pygame.K_ESCAPE:running=False
                    elif e.key in (pygame.K_BACKSPACE,pygame.K_DELETE):msg='Use the folder/file selectors; year and filename can be edited in source settings.'
            elif state=='MAP':
                if e.type==pygame.KEYDOWN:
                    if e.key==pygame.K_ESCAPE:state='IMAGES'
                    elif e.key==pygame.K_s:msg='Saved calibration' if guardar_calibracion(calibrated_path,segments) else 'Save failed'
                    elif e.key==pygame.K_r:
                        count=repair_line_continuity(segments);msg=f'R: continuity repair adjusted {count} segment(s) | press S to save'
                elif e.type==pygame.MOUSEBUTTONDOWN and e.button==1:
                    if save_map_btn.collidepoint(e.pos):msg='Saved calibration' if guardar_calibracion(calibrated_path,segments) else 'Save failed'
                    elif OFFSET_MAPA_X<=e.pos[0]<=OFFSET_MAPA_X+CANVAS_W and OFFSET_MAPA_Y<=e.pos[1]<=OFFSET_MAPA_Y+CANVAS_H:
                        ok,msg,gid=auto_ajustar_por_color(e.pos,image,segments);selected=gid if ok else None
        if state=='IMAGES':
            screen.blit(title_font.render('IMAGES',True,BLUE),(WIDTH//2-60,65))
            for i,(name,val) in enumerate(fields):
                screen.blit(font.render(name,True,TEXT),(rects[i].x,rects[i].y-25));pygame.draw.rect(screen,BOX,rects[i],border_radius=5);screen.blit(font.render(str(val)[:80],True,TEXT),(rects[i].x+8,rects[i].y+9))
            button(save_btn,BLUE,'SAVE SETTINGS',mouse);button(load_btn,GREEN,'LOAD MAP',mouse)
            screen.blit(msg_font.render('Select image directory and obstacle file, then load the map.',True,TEXT),(190,565));screen.blit(msg_font.render(msg[:110],True,TEXT),(190,600))
        else:
            screen.blit(image,(OFFSET_MAPA_X,OFFSET_MAPA_Y))
            for s in segments:
                p1,p2=calcular_puntos_px(s['data']);label=str(s['data'][4]).lower();col=(0,150,255) if 'blue' in label else (255,50,50) if 'red' in label else (255,165,0);width=6 if s['group_id']==selected else 3
                if s['group_id']==selected:col=(255,255,0)
                pygame.draw.line(screen,col,p1,p2,width)
            button(save_map_btn,GREEN,'SAVE [S]',mouse);screen.blit(msg_font.render('Click to align groups | R: repair continuity | S: save | ESC: settings',True,TEXT),(OFFSET_MAPA_X,655));screen.blit(msg_font.render(msg[:110],True,TEXT),(OFFSET_MAPA_X,630))
        pygame.display.flip();pygame.time.Clock().tick(30)
    pygame.quit()
if __name__=='__main__':main()
